# pandas 표 + matplotlib 그래프 대시보드
- `gr.File`로 파일을 업로드받아 pandas로 읽는 흐름을 익힌다.
- Gradio 출력으로 **표(DataFrame)와 그래프(matplotlib)를 동시에** 보여주는 법을 익힌다.

### 💡 강의 포인트
- Gradio는 matplotlib의 `Figure` 객체를 그대로 `outputs`에 넣으면 이미지처럼 화면에 그려줍니다 — 이미 알고 있는 시각화 코드를 거의 그대로 재사용할 수 있다는 점을 강조하면 좋습니다.
- 파일 업로드 컴포넌트(`gr.File`)는 업로드된 파일의 **경로(path)**를 문자열로 돌려준다는 점이 특징입니다.

In [1]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='Malgun Gothic')  
mpl.rc('axes', unicode_minus=False)

In [2]:
def analyze_csv(file):
    # file : 업로드된 파일의 임시 경로(문자열)가 들어옵니다.
    df = pd.read_csv(file.name, encoding="utf-8-sig")

    # 숫자형 컬럼만 골라서 간단한 요약 통계표 생성
    numeric_df = df.select_dtypes(include="number")
    summary_table = numeric_df.describe().round(2)

    # 숫자형 컬럼 중 첫 번째 컬럼으로 히스토그램을 그려서 분포 확인
    fig, ax = plt.subplots(figsize=(6, 4))
    if len(numeric_df.columns) > 0:
        first_col = numeric_df.columns[0]
        ax.hist(numeric_df[first_col], bins=10, color="skyblue", edgecolor="black")
        ax.set_title(f"{first_col} 분포")
    else:
        ax.text(0.5, 0.5, "숫자형 컬럼이 없습니다", ha="center")

    return summary_table, fig

In [ ]:
demo = gr.Interface(
    fn=analyze_csv,
    inputs=gr.File(label="CSV 파일 업로드", file_types=[".csv"]),
    outputs=[
        gr.Dataframe(label="요약 통계표"),  # DataFrame을 표 형태로 보여줌
        gr.Plot(label="분포 히스토그램"),    # matplotlib Figure를 이미지로 보여줌
    ],
    title="CSV 업로드 분석 대시보드",
    description="CSV 파일을 올리면 요약 통계와 히스토그램을 함께 보여줍니다. (Chapter01의 CSV 파일로 테스트해보세요)",
)
demo.launch()

In [5]:

# ⚠️ 실습 팁: 노트북에서 Gradio를 계속 켜두면 포트가 쌓여서 다음 셀 실행이 꼬일 수 있습니다.
# 확인이 끝나면 아래처럼 데모를 꺼주는 습관을 들이세요.
demo.close()


Closing server running on port: 7860


### ❓ 생각해볼 질문
1. 숫자형 컬럼이 여러 개인 CSV를 올리면, 지금 코드는 왜 첫 번째 컬럼으로만 히스토그램을 그릴까? 모든 숫자 컬럼에 대해 그리려면 어떻게 바꿔야 할까?
2. `gr.Dataframe`과 `gr.Plot`처럼 서로 다른 타입의 출력을 한 화면에 같이 두면 어떤 점이 편리할까?